In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
name = torch.cuda.get_device_name(0); print("GPU:", name, flush=True)
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(dst): os.symlink(p, dst)
hard = os.path.dirname(glob.glob("/kaggle/input/**/hard_neg_strict.parquet", recursive=True)[0])
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

# bge-reranker-v2-m3 уже обучена оценивать пары «запрос-кандидат», поэтому первый этап
# на LLM-парах пропускаем: модель приходит умеющей сравнивать, ей нужна только специфика
# товаров. Шаг 1e-5, а не 3e-5: на 3e-5 у нас дважды разваливались крупные основы в fp16.
BASE = "BAAI/bge-reranker-v2-m3"
sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            # Первый этап отключается через --epochs 0, а не через --resume-from:
            # последний включает local_files_only, и модель перестаёт скачиваться с Hub.
            "--base-model", BASE, "--epochs", "0",
            "--hard-negatives", f"{hard}/hard_neg_strict.parquet",
            "--batch-size", "32", "--max-length", "256",
            "--human-epochs", "1", "--human-learning-rate", "1e-5",
            "--output", "/kaggle/working/ce_bge"]
from src.train_ce_large import main
main()
